# ML Modeling Metrics

Companion notebook for the ML Modeling Metrics learning session.

Covers confusion matrix, precision, recall, F1, threshold sweeps, PR curves, calibration bins, and trajectory errors.

In [ ]:
import torch
import matplotlib.pyplot as plt
torch.manual_seed(0)

## Binary metrics from first principles

Positive class: a pedestrian crosses.

In [ ]:
def binary_metrics(scores, labels, threshold=0.5):
    pred = scores >= threshold
    labels = labels.bool()
    tp = (pred & labels).sum().item()
    fp = (pred & ~labels).sum().item()
    fn = (~pred & labels).sum().item()
    tn = (~pred & ~labels).sum().item()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    accuracy = (tp + tn) / max(tp + fp + fn + tn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, precision=precision, recall=recall, accuracy=accuracy, f1=f1)

# Imbalanced toy data: 5% positives.
n = 1000
labels = (torch.rand(n) < 0.05)
# Scores: positives tend to be higher, but overlap with negatives.
scores = torch.where(labels, torch.rand(n) * 0.6 + 0.35, torch.rand(n) * 0.7)

for th in [0.2, 0.5, 0.8]:
    print('threshold', th, binary_metrics(scores, labels, th))

## Threshold sweep

Lower threshold usually increases recall and decreases precision.

In [ ]:
thresholds = torch.linspace(0, 1, 51)
precisions, recalls, f1s = [], [], []
for th in thresholds:
    m = binary_metrics(scores, labels, float(th))
    precisions.append(m['precision'])
    recalls.append(m['recall'])
    f1s.append(m['f1'])

plt.figure(figsize=(6, 4))
plt.plot(thresholds, precisions, label='precision')
plt.plot(thresholds, recalls, label='recall')
plt.plot(thresholds, f1s, label='F1')
plt.xlabel('threshold'); plt.legend(); plt.title('Threshold tradeoff'); plt.show()

best_i = int(torch.tensor(f1s).argmax())
print('best F1 threshold:', float(thresholds[best_i]), 'F1:', f1s[best_i])

## PR curve by sorting scores

Precision/recall curve asks: if I review the top-scored examples, how many positives do I recover and how noisy is the list?

In [ ]:
order = torch.argsort(scores, descending=True)
sorted_labels = labels[order].float()
cum_tp = torch.cumsum(sorted_labels, dim=0)
k = torch.arange(1, n + 1).float()
precision_at_k = cum_tp / k
recall_at_k = cum_tp / sorted_labels.sum().clamp_min(1)

plt.figure(figsize=(5, 4))
plt.plot(recall_at_k, precision_at_k)
plt.xlabel('recall'); plt.ylabel('precision'); plt.title('PR curve'); plt.show()

for topk in [10, 50, 100]:
    print(f'precision@{topk}:', float(precision_at_k[topk - 1]), 'recall@k:', float(recall_at_k[topk - 1]))

## Calibration bins

If the model says 0.8, is the event actually positive about 80% of the time?

In [ ]:
def calibration_bins(scores, labels, bins=10):
    edges = torch.linspace(0, 1, bins + 1)
    rows = []
    for i in range(bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (scores >= lo) & (scores < hi if i < bins - 1 else scores <= hi)
        if mask.any():
            conf = scores[mask].mean().item()
            acc = labels[mask].float().mean().item()
            rows.append((float(lo), float(hi), int(mask.sum()), conf, acc))
    return rows

rows = calibration_bins(scores, labels)
for row in rows:
    print('bin [%.1f, %.1f], n=%d, avg_conf=%.3f, empirical_pos=%.3f' % row)

## Regression / trajectory metrics

In [ ]:
def ade(pred, target):
    return torch.norm(pred - target, dim=-1).mean()

def fde(pred, target):
    return torch.norm(pred[:, -1] - target[:, -1], dim=-1).mean()

T = 20
t = torch.linspace(0, 1, T)
target = torch.stack([5 * t, torch.zeros_like(t)], dim=-1)[None]
pred = target + 0.2 * torch.randn_like(target)
print('ADE:', ade(pred, target).item())
print('FDE:', fde(pred, target).item())

plt.figure(figsize=(5, 4))
plt.plot(target[0, :, 0], target[0, :, 1], label='target')
plt.plot(pred[0, :, 0], pred[0, :, 1], label='pred')
plt.legend(); plt.axis('equal'); plt.title('Trajectory error'); plt.show()

Interview drills:

1. Why is accuracy bad for rare pedestrian crossing?
2. What threshold would you choose if false negatives are much worse than false positives?
3. Why is PR-AUC usually better than ROC-AUC for rare events?
4. Why are ADE/FDE not enough for simulation?